# 11 — Machine translation, decoding strategies, BLEU, ROUGE, and generation controls

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Machine translation is sequence-to-sequence:

```text
source language sentence → target language sentence
```

This notebook covers toy translation, decoding strategies, BLEU, and generation controls that also apply to LLMs.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

## Toy dictionary translation baseline

This is not real MT. It demonstrates why word translation alone is insufficient: real MT also needs reordering, grammar, context, and pragmatics.

In [ ]:
lexicon = {
    "i": "je",
    "love": "aime",
    "the": "le",
    "dashboard": "tableau de bord",
    "app": "application",
    "crashes": "plante",
    "invoice": "facture",
}

def dictionary_translate(sentence):
    return " ".join(lexicon.get(w.lower(), f"[{w}]") for w in sentence.split())

print(dictionary_translate("I love the dashboard"))
print(dictionary_translate("The app crashes"))

## Decoding utilities

Generation models output logits. Decoding turns logits into tokens.

In [ ]:
import numpy as np
import math

vocab = ["the", "app", "crashes", "works", "great", ".", "<eos>"]
logits = np.array([2.0, 1.5, 0.7, 0.6, 0.4, 0.2, -0.5])

def softmax(x, temperature=1.0):
    x = np.array(x) / temperature
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def greedy_decode(logits):
    return int(np.argmax(logits))

def sample_decode(logits, temperature=1.0):
    p = softmax(logits, temperature)
    return int(np.random.choice(len(p), p=p))

def top_k_decode(logits, k=3, temperature=1.0):
    idx = np.argsort(logits)[-k:]
    p = softmax(logits[idx], temperature)
    return int(np.random.choice(idx, p=p))

def top_p_decode(logits, p_threshold=0.9, temperature=1.0):
    p = softmax(logits, temperature)
    order = np.argsort(p)[::-1]
    cumulative = np.cumsum(p[order])
    keep = order[cumulative <= p_threshold]
    if len(keep) == 0:
        keep = order[:1]
    elif keep[-1] != order[min(len(keep), len(order)-1)]:
        keep = order[:len(keep)+1]
    p_keep = p[keep] / p[keep].sum()
    return int(np.random.choice(keep, p=p_keep))

for name, fn in [
    ("greedy", lambda: greedy_decode(logits)),
    ("sample", lambda: sample_decode(logits, temperature=1.0)),
    ("top_k", lambda: top_k_decode(logits, k=3)),
    ("top_p", lambda: top_p_decode(logits, p_threshold=0.8)),
]:
    print(name, vocab[fn()])

## Beam search over a toy next-token model

Beam search keeps the best B partial sequences. It is common in translation and summarisation.

In [ ]:
# Toy conditional probabilities
next_probs = {
    "<s>": {"the": 0.6, "app": 0.3, "great": 0.1},
    "the": {"app": 0.7, "dashboard": 0.2, "<eos>": 0.1},
    "app": {"crashes": 0.5, "works": 0.4, "<eos>": 0.1},
    "dashboard": {"works": 0.6, "<eos>": 0.4},
    "crashes": {".": 0.8, "<eos>": 0.2},
    "works": {"great": 0.6, ".": 0.3, "<eos>": 0.1},
    "great": {".": 0.7, "<eos>": 0.3},
    ".": {"<eos>": 1.0},
}

def beam_search(beam_size=3, max_len=6):
    beams = [(["<s>"], 0.0)]
    for _ in range(max_len):
        candidates = []
        for seq, logp in beams:
            last = seq[-1]
            if last == "<eos>":
                candidates.append((seq, logp))
                continue
            for tok, p in next_probs.get(last, {"<eos>": 1.0}).items():
                candidates.append((seq + [tok], logp + math.log(p)))
        beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_size]
    return beams

for seq, score in beam_search():
    print(seq, score)

## BLEU from scratch

BLEU uses clipped n-gram precision plus brevity penalty. It rewards surface overlap and penalizes overly short translations.

In [ ]:
from collections import Counter

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def bleu(candidate, references, max_n=4):
    cand = candidate.split()
    refs = [r.split() for r in references]
    precisions = []
    for n in range(1, max_n+1):
        cand_counts = Counter(ngrams(cand, n))
        if not cand_counts:
            precisions.append(0)
            continue
        max_ref_counts = Counter()
        for ref in refs:
            ref_counts = Counter(ngrams(ref, n))
            for gram, count in ref_counts.items():
                max_ref_counts[gram] = max(max_ref_counts[gram], count)
        clipped = sum(min(count, max_ref_counts[gram]) for gram, count in cand_counts.items())
        precisions.append(clipped / sum(cand_counts.values()))
    # Smoothing to avoid zero for demo
    precisions = [max(p, 1e-9) for p in precisions]
    geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    ref_lens = [len(r) for r in refs]
    cand_len = len(cand)
    closest_ref_len = min(ref_lens, key=lambda rl: abs(rl - cand_len))
    bp = 1.0 if cand_len > closest_ref_len else math.exp(1 - closest_ref_len / max(cand_len, 1))
    return bp * geo_mean

candidate = "the app crashes"
references = ["the application crashes", "the app is crashing"]
bleu(candidate, references)

## ROUGE vs BLEU

- BLEU: precision-oriented, common in MT
- ROUGE: recall-oriented, common in summarisation
- METEOR: alignment + precision/recall + stemming/synonyms
- BLEURT: learned BERT-based metric correlated with human judgments

Always pair automatic metrics with human checks for adequacy and fluency.

## Product note

Generation controls matter:

| Control | Effect |
|---|---|
| low temperature | deterministic, safer |
| high temperature | creative, risky |
| beam search | strong for translation/summarisation |
| top-p/top-k | useful for open-ended generation |
| constraints | useful for JSON/schema/tool outputs |